# Lab 7: Parallel Programming with Python

Welcome to Lab 7 of Programming Methods! Lab 6 measured complexity and found the bottlenecks; this lab asks the natural next question - can we run several things **at once**? We cover the concurrency decision tree, the I/O-bound vs. CPU-bound distinction, the GIL, and three main paradigms: threading, multiprocessing, and `asyncio`.

Create a new branch for this Lab (e.g., `git checkout -b lab-07-parallel`). Then, complete the following exercises in **this** Jupyter Notebook. Once you are done, **commit your work** using the proper Git commands.

The solutions for this lab are included after each exercise.

## Learning objectives

By the end of this class, you should be able to:

- Use the **concurrency decision tree** to decide whether a task calls for threads, processes, or `asyncio`.
- Explain the **GIL** and the **I/O-bound vs. CPU-bound** distinction that determines which paradigm actually helps.
- Implement **`ThreadPoolExecutor`** to parallelize I/O-bound tasks like API requests and Ollama inferences.
- Use **`ProcessPoolExecutor`** to bypass the GIL and accelerate CPU-bound text processing.
- Use **`asyncio`** as a lighter-weight alternative to threads for overlapping many I/O-bound waits.

## Setup

Run this cell first. It locates the repository, defines the reviews file path, and imports `count_duplicates_fast` from `pm_labs.timing`. This benchmark workload uses the same algorithm as Lab 6; keeping it in an importable Python module lets process-pool workers load it in a fresh interpreter.

In [ ]:
import asyncio
import sys
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from pathlib import Path

for PROJECT_ROOT in (Path.cwd(), *Path.cwd().parents):
    if (PROJECT_ROOT / "src" / "pm_labs").is_dir():
        break
else:
    raise FileNotFoundError("Run this notebook from within the course repository.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from pm_labs import OllamaClient, WorldBankClient, retry, timer
from pm_labs.timing import count_duplicates_fast

REVIEWS_PATH = PROJECT_ROOT / "data" / "reviews.txt"

wb_client = WorldBankClient()
ollama_client = OllamaClient(model="llama3.2:1b")

---
## Choosing a concurrency strategy

Before reaching for threads, processes, or `asyncio`, ask two questions: **(1)** Does the task spend most of its time *waiting* (network, disk) or *computing*? **(2)** If waiting, do I need to manage thousands of tasks at once, or just a handful?

Use the table below as a reference when choosing how to run a workload.

| Strategy | When to use it | Example | Main caveat |
| --- | --- | --- | --- |
| Sequential | A small or quick workload where coordination costs more than it saves | Counting duplicates in a short list | No overlap or multicore execution |
| Threads (`ThreadPoolExecutor`) | Blocking I/O with a manageable number of concurrent requests | World Bank API calls or a few Ollama prompts | Limit workers to server capacity; threads do not parallelize pure-Python CPU work under the GIL |
| Processes (`ProcessPoolExecutor`) | Substantial CPU-bound pure-Python work that can be split into independent chunks | Counting duplicates separately within large chunks | Startup and data transfer cost time; workers must be importable and inputs/results picklable |
| `asyncio` with an async-compatible client | Many concurrent I/O waits | Thousands of URL requests or Ollama prompts, with bounded concurrency | Blocking calls stall the event loop unless offloaded; `asyncio.to_thread` still uses a thread pool |

There is no universal task-count cutoff. Measure the workload and account for library support, concurrency limits, and coordination overhead.

**Why this matters:** the decision tree isn't about which paradigm is "fastest" in the abstract - it's about matching the tool to *why* a task is slow. Applying the wrong one (e.g. threads on a CPU-bound task) adds concurrency overhead without addressing the bottleneck.

---
## Exercise 1: I/O-bound work with `ThreadPoolExecutor`

Threads help exactly when a task spends most of its time **waiting**: the GIL is released during blocking I/O (network, disk), so other threads can run during that wait. We parallelize two I/O-bound tasks from earlier labs: World Bank API requests (Lab 3) and Ollama inferences (Lab 3/5).

### 1a. Parallelizing API requests

Fetch populations for many more countries than Lab 6's 8, sequentially vs. with `ThreadPoolExecutor.map`, timing both with `@timer`.

In [ ]:
country_codes = ["pt", "es", "fr", "de", "it", "nl", "be", "gr", "pl", "se", "no", "dk", "fi", "ie", "at", "ch"]


def fetch_one(code: str) -> int:
    return wb_client.get_population(code).value


@timer
def fetch_sequential(codes: list[str]) -> list[int]:
    pass


@timer
def fetch_threaded(codes: list[str], max_workers: int = 16) -> list[int]:
    pass


sequential_results = fetch_sequential(country_codes)
threaded_results = fetch_threaded(country_codes)
assert sequential_results == threaded_results

### 1b. Parallelizing Ollama inferences

Send several prompts to `ollama_client` sequentially vs. with `ThreadPoolExecutor.map`. Wrap each call with the **`@retry`** decorator (Lab 5, now in `pm_labs.decorators`) so a single slow/failed inference doesn't sink the whole batch.

In [ ]:
prompts = [
    "In one sentence, what is a thread?",
    "In one sentence, what is a process?",
    "In one sentence, what is the GIL?",
    "In one sentence, what is asyncio?",
]


@retry
def ask_ollama(prompt: str) -> str:
    return ollama_client.generate_validated(prompt).response


@timer
def ask_sequential(all_prompts: list[str]) -> list[str]:
    pass


@timer
def ask_threaded(all_prompts: list[str], max_workers: int = 4) -> list[str]:
    pass


sequential_answers = ask_sequential(prompts)
threaded_answers = ask_threaded(prompts)
for prompt, answer in zip(prompts, threaded_answers):
    print(f"- {prompt}\n  {answer}\n")

**Why this matters:** both tasks are I/O-bound (waiting on a network socket to the World Bank API or to the local Ollama server), so `ThreadPoolExecutor` overlaps their waits and gives a real speedup even though only one thread runs Python bytecode at a time. The speedup scales roughly with `max_workers` up to the point where the server (or your machine's network/CPU) becomes the bottleneck instead.

---
## Exercise 2: CPU-bound work with `ProcessPoolExecutor`

CPU-bound work (in-memory computation, no waiting) never releases the GIL for any meaningful stretch, so threads can't run it in parallel - only separate **processes**, each with its own interpreter and GIL, can. We reuse `count_duplicates_fast` (Setup, same as Lab 6) as a CPU-bound text-processing task, splitting a large dataset into chunks.

In [ ]:
with open(REVIEWS_PATH) as f:
    all_reviews = [line.strip() for line in f]

large_dataset = (all_reviews * 20)[:20_000]
chunk_size = 5_000
chunks = [large_dataset[i:i + chunk_size] for i in range(0, len(large_dataset), chunk_size)]


@timer
def count_sequential(data_chunks: list[list[str]]) -> list[int]:
    pass


@timer
def count_threaded(data_chunks: list[list[str]], max_workers: int = 4) -> list[int]:
    pass


@timer
def count_parallel(data_chunks: list[list[str]], max_workers: int = 4) -> list[int]:
    pass


sequential_counts = count_sequential(chunks)
threaded_counts = count_threaded(chunks)
process_counts = count_parallel(chunks)
assert sequential_counts == threaded_counts == process_counts

**Why this matters:** in standard GIL-enabled CPython, `count_threaded` cannot execute this pure-Python computation on multiple cores, so thread coordination can make it slower than `count_sequential`. `count_parallel` uses separate interpreters and can run on multiple cores, but process startup and data transfer may outweigh the benefit for small chunks. The decision tree recommends **CPU-bound -> processes**, not threads; timing determines whether the workload is large enough to benefit.

---
## Exercise 3: `asyncio` for many concurrent I/O-bound tasks

Threads work well for a handful of I/O-bound tasks, but each OS thread has real memory/scheduling overhead - with thousands of concurrent waits, `asyncio` is cheaper because a single thread cooperatively switches between tasks whenever one is waiting. `asyncio.to_thread` lets us reuse an existing *synchronous* function (like `fetch_one`) inside `async`/`await` code without rewriting it as a native async client.

In [ ]:
async def fetch_all_async(codes: list[str]) -> list[int]:
    pass


start = time.perf_counter()
async_results = await fetch_all_async(country_codes)
print(f"fetch_all_async took {time.perf_counter() - start:.3f}s")
assert async_results == sequential_results

**Why this matters:** `fetch_all_async` uses `asyncio.to_thread` to run blocking requests in a thread pool, so its performance may be similar to `fetch_threaded`. It demonstrates how to integrate synchronous code with an event loop, not native asynchronous networking. For thousands of I/O waits, an async-compatible client can avoid one operating-system thread per waiting request. Both approaches still need concurrency limits that respect server capacity and rate limits. The decision tree considers *task count* as well as whether work is I/O-bound or CPU-bound.

---
## Wrap-up questions (choose one answer for each)

1. Following the decision tree, which combination best matches these tasks: 8 blocking World Bank API calls, mostly waiting on the network; CPU-heavy duplicate counting in pure Python across 4 in-memory chunks; 10,000 concurrent URL requests using an async-compatible HTTP client?
   - A. Processes; threads; `asyncio`.
   - B. Threads; processes; `asyncio`.
   - C. `asyncio`; processes; threads.
   - D. Threads; `asyncio`; processes.
2. Why can `ThreadPoolExecutor` show little speedup for CPU-bound Python code?
   - A. Threads cannot share data.
   - B. The GIL limits Python bytecode execution to one thread at a time.
   - C. Threads always run on separate machines.
   - D. Thread pools turn CPU work into network work.
3. Why can `asyncio` be a better fit than threads for thousands of I/O waits?
   - A. It gives each task its own process.
   - B. It removes all network latency.
   - C. It can manage many waiting tasks without one operating-system thread per task.
   - D. It bypasses the GIL for CPU-bound work.
4. What does `asyncio.to_thread` do when called with a synchronous function?
   - A. Runs a synchronous function in a worker thread so it can be awaited.
   - B. Converts a synchronous function into a native asynchronous network client.
   - C. Runs Python code in a separate process.
   - D. Makes CPU-bound code execute in parallel across cores.

> #### 📚 Further reading
> - Ramalho, *Fluent Python* (2nd ed.), **concurrency chapters** on threads, processes, and `asyncio` - the GIL, when each paradigm helps, and `concurrent.futures` as the high-level entry point used throughout this lab.
> - Python docs, [`concurrent.futures`](https://docs.python.org/3/library/concurrent.futures.html) - managing I/O-bound work with `ThreadPoolExecutor` and CPU-bound work with `ProcessPoolExecutor`.
> - Python docs, [`asyncio`](https://docs.python.org/3/library/asyncio.html) and [`asyncio.to_thread`](https://docs.python.org/3/library/asyncio.html#asyncio.to_thread) - event loops and running blocking functions in worker threads.
> - Python docs, [`multiprocessing` programming guidelines](https://docs.python.org/3/library/multiprocessing.html#programming-guidelines) - importability and pickling requirements for worker functions such as `count_duplicates_fast` when using `ProcessPoolExecutor`.
> - Lab 6's profiling exercise - the `cProfile`/`tracemalloc` results that motivated profiling and empirical performance measurement.